<a href="https://colab.research.google.com/github/prachipatil-pm/bovine-mastitis-ai/blob/main/SIH_Bovine_Mastitis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install streamlit scikit-learn pandas joblib plotly -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 55.4 MB/s eta 0:00:00


In [2]:
import pandas as pd

data = {
    "Cow_ID": [
        "COW001","COW002","COW003","COW004","COW005",
        "COW006","COW007","COW008","COW009","COW010",
        "COW011","COW012","COW013","COW014","COW015",
        "COW016","COW017","COW018","COW019","COW020"
    ],

    "Breed_Type": [
        "Holstein Cross","Jersey Cross","HF Cross","Jersey Cross","HF Cross",
        "Gir Cross","HF Cross","Jersey Cross","HF Cross","Gir Cross",
        "HF Cross","Jersey Cross","HF Cross","Jersey Cross","HF Cross",
        "Gir Cross","HF Cross","Jersey Cross","HF Cross","Gir Cross"
    ],

    "Milk_EC_mS_cm": [
        4.2,4.5,4.7,4.9,5.1,
        5.0,5.6,5.8,6.0,6.1,
        6.3,6.4,6.7,6.9,7.1,
        7.3,7.5,7.7,8.0,8.2
    ],

    "Milk_Yield_L_day": [
        0.82,0.76,0.88,0.71,0.84,
        0.79,0.68,0.73,0.69,0.77,
        0.65,0.70,0.61,0.58,0.55,
        0.60,0.52,0.49,0.46,0.43
    ],

    "SCC_thousand_cells_mL": [
        145,180,165,210,195,
        225,310,360,420,390,
        480,510,620,710,820,
        760,910,980,1120,1250
    ],

    "Risk_Label": [
        "Low","Low","Low","Low","Low","Low",
        "Medium","Medium","Medium","Medium","Medium","Medium",
        "High","High","High","High","High","High","High","High"
    ]
}

df = pd.DataFrame(data)

df

,Cow_ID,Breed_Type,Milk_EC_mS_cm,Milk_Yield_L_day,SCC_thousand_cells_mL,Risk_Label
0,COW001,Holstein Cross,4.2,0.82,145,Low
1,COW002,Jersey Cross,4.5,0.76,180,Low
2,COW003,HF Cross,4.7,0.88,165,Low
3,COW004,Jersey Cross,4.9,0.71,210,Low
4,COW005,HF Cross,5.1,0.84,195,Low
5,COW006,Gir Cross,5.0,0.79,225,Low
6,COW007,HF Cross,5.6,0.68,310,Medium
7,COW008,Jersey Cross,5.8,0.73,360,Medium
8,COW009,HF Cross,6.0,0.69,420,Medium
9,COW010,Gir Cross,6.1,0.77,390,Medium


In [3]:
df.to_csv("mastitis_20_cows.csv", index=False)

print("20-cow dataset created successfully!")

20-cow dataset created successfully!


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

X = df[["Milk_EC_mS_cm"]]
y = df["Risk_Label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("Model trained successfully!")
print("Demo dataset accuracy:", round(accuracy * 100, 2), "%")

Model trained successfully!
Demo dataset accuracy: 100.0 %


In [5]:
import joblib

joblib.dump(model, "mastitis_model.pkl")

print("ML model saved successfully!")

ML model saved successfully!


In [6]:
test_values = [4.5, 5.8, 6.4, 7.2, 8.0]

for value in test_values:
    result = model.predict([[value]])[0]
    print("EC:", value, "→", result)

EC: 4.5 → Low
EC: 5.8 → Medium
EC: 6.4 → Medium
EC: 7.2 → High
EC: 8.0 → High


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [7]:
%%writefile app.py

import streamlit as st
import pandas as pd
import joblib
import plotly.express as px

# --------------------------------------------------
# PAGE CONFIGURATION
# --------------------------------------------------

st.set_page_config(
    page_title="Bovine Mastitis AI",
    page_icon="🐄",
    layout="wide"
)

# --------------------------------------------------
# LOAD DATA AND MODEL
# --------------------------------------------------

df = pd.read_csv("mastitis_20_cows.csv")
model = joblib.load("mastitis_model.pkl")

# --------------------------------------------------
# HEADER
# --------------------------------------------------

st.title("🐄 Bovine Mastitis AI")
st.subheader(
    "AI-Based Early Risk Monitoring System for Dairy Cows"
)

st.info(
    "This prototype provides an early-risk indication using "
    "milk electrical conductivity. Veterinary examination is "
    "recommended for high-risk cases."
)

# --------------------------------------------------
# SIDEBAR
# --------------------------------------------------

st.sidebar.header("🐄 Cow Monitoring")

selected_cow = st.sidebar.selectbox(
    "Select Cow",
    df["Cow_ID"].tolist()
)

# --------------------------------------------------
# SELECTED COW
# --------------------------------------------------

cow = df[df["Cow_ID"] == selected_cow].iloc[0]

# --------------------------------------------------
# KPI CARDS
# --------------------------------------------------

total_cows = len(df)
low_count = len(df[df["Risk_Label"] == "Low"])
medium_count = len(df[df["Risk_Label"] == "Medium"])
high_count = len(df[df["Risk_Label"] == "High"])

col1, col2, col3, col4 = st.columns(4)

col1.metric("🐄 Cows Monitored", total_cows)
col2.metric("🟢 Low Risk", low_count)
col3.metric("🟠 Medium Risk", medium_count)
col4.metric("🔴 High Risk", high_count)

st.divider()

# --------------------------------------------------
# SELECTED COW INFORMATION
# --------------------------------------------------

st.header("🔍 Selected Cow Analysis")

c1, c2, c3, c4 = st.columns(4)

c1.metric(
    "Cow ID",
    cow["Cow_ID"]
)

c2.metric(
    "Breed",
    cow["Breed_Type"]
)

c3.metric(
    "Milk EC",
    f'{cow["Milk_EC_mS_cm"]} mS/cm'
)

c4.metric(
    "Milk Yield",
    f'{cow["Milk_Yield_L_day"]} L/day'
)

# --------------------------------------------------
# ML PREDICTION
# --------------------------------------------------

ec_value = float(cow["Milk_EC_mS_cm"])

prediction = model.predict(
    pd.DataFrame(
        {"Milk_EC_mS_cm": [ec_value]}
    )
)[0]

# --------------------------------------------------
# RISK DISPLAY
# --------------------------------------------------

st.subheader("🤖 AI Early-Risk Assessment")

if prediction == "Low":

    st.success(
        "🟢 LOW RISK — Continue routine monitoring."
    )

elif prediction == "Medium":

    st.warning(
        "🟠 MEDIUM RISK — Monitor the cow closely "
        "and consider veterinary examination."
    )

else:

    st.error(
        "🔴 HIGH RISK — Veterinary examination recommended."
    )

# --------------------------------------------------
# GRAPH
# --------------------------------------------------

st.header("📊 Milk Electrical Conductivity Monitoring")

fig = px.bar(
    df,
    x="Cow_ID",
    y="Milk_EC_mS_cm",
    color="Risk_Label",
    title="Milk EC Across Monitored Cows",
    labels={
        "Milk_EC_mS_cm": "Milk EC (mS/cm)",
        "Cow_ID": "Cow"
    }
)

st.plotly_chart(
    fig,
    use_container_width=True
)

# --------------------------------------------------
# COMPLETE COW TABLE
# --------------------------------------------------

st.header("📋 Cow Monitoring Records")

display_df = df.copy()

st.dataframe(
    display_df,
    use_container_width=True,
    hide_index=True
)

# --------------------------------------------------
# FOOTER
# --------------------------------------------------

st.divider()

st.caption(
    "SIH 2026 Prototype | Synthetic demonstration dataset | "
    "EC-based early-risk indication, not a clinical diagnosis"
)

Writing app.py


In [8]:
!pkill -f streamlit || true

^C


In [9]:
!streamlit run app.py --server.port 8503 --server.address 0.0.0.0 > /content/sih_dashboard.log 2>&1 &

In [10]:
from google.colab.output import eval_js

url = eval_js("google.colab.kernel.proxyPort(8503)")
print(url)

https://8503-m-s-kkb-ase1a0-1k6abnsaeb4u8-a.asia-east1-0.prod.colab.dev


In [11]:
!tail -100 /content/sih_dashboard.log



2026-09-08 17:42:31.053 Uvicorn server started on 0.0.0.0:8503

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8503
  Network URL: http://172.28.0.12:8503
  External URL: http://34.81.61.64:8503

2026-09-08 17:43:08.978 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8503-m-s-kkb-ase1a0-1k6abnsaeb4u8-a.asia-east1-0.prod.colab.dev, host=m-s-kkb-ase1a0-1k6abnsaeb4u8.asia-east1-a.c.codatalab-user-runtimes.internal:8007
2026-09-08 17:43:09.908 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8503-m-s-kkb-ase1a0-1k6abnsaeb4u8-a.asia-east1-0.prod.colab.dev, host=m-s-kkb-ase1a0-1k6abnsaeb4u8.asia-east1-a.c.codatalab-user-runtimes.internal:8007
2026-09-08 17:43:11.253 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8503-m-s-kkb-ase1a0-1k6abnsaeb4u8-a.asia-east1-0.prod.colab.dev, host=m-s-kkb-ase1a0-1k6abnsaeb4u8.asia-east1-a.c.codatalab-user-run

In [12]:
!pkill -9 -f "streamlit run" || true

^C


In [13]:
!streamlit run app.py --server.port 8503 --server.address 0.0.0.0 --server.enableCORS false --server.enableXsrfProtection false > /content/sih_dashboard.log 2>&1 &

In [14]:
from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(8503)")
print(url)

https://8503-m-s-kkb-ase1a0-1k6abnsaeb4u8-a.asia-east1-0.prod.colab.dev


In [15]:
%%writefile app.py

import streamlit as st
import pandas as pd
import joblib
import plotly.express as px

# -----------------------------
# PAGE
# -----------------------------

st.set_page_config(
    page_title="Bovine Mastitis AI",
    page_icon="🐄",
    layout="wide"
)

# -----------------------------
# LOAD DATA + MODEL
# -----------------------------

df = pd.read_csv("mastitis_20_cows.csv")
model = joblib.load("mastitis_model.pkl")

# -----------------------------
# HEADER
# -----------------------------

st.title("🐄 Bovine Mastitis AI")
st.markdown(
    "### AI-Based Early-Risk Monitoring System for Dairy Cows"
)

st.info(
    "This prototype uses milk electrical conductivity (EC) "
    "to provide an early-risk indication. High-risk cases "
    "should be followed by veterinary examination."
)

# -----------------------------
# SIDEBAR
# -----------------------------

st.sidebar.title("🐄 Cow Monitoring")

selected_cow = st.sidebar.selectbox(
    "Select Cow",
    df["Cow_ID"].tolist()
)

st.sidebar.divider()

st.sidebar.markdown("### 📡 Sensor Input")

sensor_ec = st.sidebar.number_input(
    "Milk EC (mS/cm)",
    min_value=3.0,
    max_value=10.0,
    value=float(
        df[df["Cow_ID"] == selected_cow]["Milk_EC_mS_cm"].iloc[0]
    ),
    step=0.1
)

analyze = st.sidebar.button(
    "🔍 Analyze Cow",
    use_container_width=True
)

# -----------------------------
# KPI SECTION
# -----------------------------

total_cows = len(df)

low = len(df[df["Risk_Label"] == "Low"])
medium = len(df[df["Risk_Label"] == "Medium"])
high = len(df[df["Risk_Label"] == "High"])

st.subheader("📊 Farm Overview")

c1, c2, c3, c4 = st.columns(4)

c1.metric("🐄 Cows Monitored", total_cows)
c2.metric("🟢 Low Risk", low)
c3.metric("🟠 Medium Risk", medium)
c4.metric("🔴 High Risk", high)

st.divider()

# -----------------------------
# SELECTED COW
# -----------------------------

cow = df[df["Cow_ID"] == selected_cow].iloc[0]

st.subheader("🔍 Selected Cow")

c1, c2, c3, c4 = st.columns(4)

c1.metric(
    "Cow ID",
    cow["Cow_ID"]
)

c2.metric(
    "Breed",
    cow["Breed_Type"]
)

c3.metric(
    "Milk EC",
    f'{cow["Milk_EC_mS_cm"]} mS/cm'
)

c4.metric(
    "Milk Yield",
    f'{cow["Milk_Yield_L_day"]} L/day'
)

# -----------------------------
# AI ANALYSIS
# -----------------------------

st.divider()

st.subheader("🤖 AI Early-Risk Analysis")

input_data = pd.DataFrame({
    "Milk_EC_mS_cm": [sensor_ec]
})

prediction = model.predict(input_data)[0]

probabilities = model.predict_proba(input_data)[0]

classes = model.classes_

probability = probabilities[
    list(classes).index(prediction)
] * 100

# -----------------------------
# RISK DISPLAY
# -----------------------------

if prediction == "Low":

    st.success(
        "🟢 LOW RISK\n\n"
        "Continue routine monitoring."
    )

elif prediction == "Medium":

    st.warning(
        "🟠 MEDIUM RISK\n\n"
        "Monitor the cow closely and consider veterinary examination."
    )

else:

    st.error(
        "🔴 HIGH RISK\n\n"
        "Veterinary examination recommended."
    )

# -----------------------------
# AI RESULT CARDS
# -----------------------------

r1, r2, r3 = st.columns(3)

r1.metric(
    "Sensor EC",
    f"{sensor_ec:.1f} mS/cm"
)

r2.metric(
    "AI Risk Category",
    prediction
)

r3.metric(
    "Model Score",
    f"{probability:.1f}%"
)

st.caption(
    "Model score is a demonstration output from the prototype model "
    "and is NOT a clinically validated probability."
)

# -----------------------------
# EC GRAPH
# -----------------------------

st.divider()

st.subheader("📈 Milk Electrical Conductivity")

fig = px.bar(
    df,
    x="Cow_ID",
    y="Milk_EC_mS_cm",
    color="Risk_Label",
    title="Milk EC Across Monitored Cows",
    labels={
        "Milk_EC_mS_cm": "Milk EC (mS/cm)",
        "Cow_ID": "Cow"
    }
)

fig.add_hline(
    y=sensor_ec,
    line_dash="dash",
    annotation_text="Selected Sensor Reading"
)

st.plotly_chart(
    fig,
    use_container_width=True
)

# -----------------------------
# HIGH-RISK ALERTS
# -----------------------------

st.divider()

st.subheader("🚨 Attention Required")

high_risk_cows = df[df["Risk_Label"] == "High"]

if len(high_risk_cows) > 0:

    st.error(
        f"{len(high_risk_cows)} cows currently require attention."
    )

    st.dataframe(
        high_risk_cows[
            [
                "Cow_ID",
                "Breed_Type",
                "Milk_EC_mS_cm",
                "SCC_thousand_cells_mL",
                "Risk_Label"
            ]
        ],
        use_container_width=True,
        hide_index=True
    )

# -----------------------------
# COMPLETE DATA
# -----------------------------

st.divider()

st.subheader("📋 Farm Monitoring Records")

st.dataframe(
    df,
    use_container_width=True,
    hide_index=True
)

# -----------------------------
# DOWNLOAD
# -----------------------------

csv_data = df.to_csv(index=False)

st.download_button(
    label="📥 Download Farm Data",
    data=csv_data,
    file_name="mastitis_farm_records.csv",
    mime="text/csv",
    use_container_width=True
)

# -----------------------------
# FOOTER
# -----------------------------

st.divider()

st.caption(
    "SIH 2026 Prototype | Synthetic demonstration dataset | "
    "EC-based early-risk indication | Not a clinical diagnosis"
)

Overwriting app.py


In [16]:
!pkill -9 -f "streamlit run" || true

^C


In [17]:
!streamlit run app.py --server.port 8503 --server.address 0.0.0.0 --server.enableCORS false --server.enableXsrfProtection false > /content/sih_dashboard.log 2>&1 &

In [18]:
from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(8503)")
print(url)

https://8503-m-s-kkb-ase1a0-1k6abnsaeb4u8-a.asia-east1-0.prod.colab.dev


In [19]:
%%writefile app.py

import streamlit as st
import pandas as pd
import joblib
import plotly.express as px

# PAGE
st.set_page_config(
    page_title="Bovine Mastitis AI",
    page_icon="🐄",
    layout="wide"
)

# LOAD
df = pd.read_csv("mastitis_20_cows.csv")
model = joblib.load("mastitis_model.pkl")

# HEADER
st.title("🐄 Bovine Mastitis AI")
st.subheader("AI-Based Early-Risk Monitoring System for Dairy Cows")

st.info(
    "This prototype provides an early-risk indication using "
    "milk electrical conductivity (EC). High-risk cases should "
    "be followed by veterinary examination."
)

# ------------------------------------------------
# COW SELECTION
# ------------------------------------------------

st.header("🐄 Select Cow for Monitoring")

selected_cow = st.selectbox(
    "Choose a Cow ID",
    options=df["Cow_ID"].tolist(),
    index=0
)

cow = df[df["Cow_ID"] == selected_cow].iloc[0]

st.success(f"Currently monitoring: **{selected_cow}**")

# ------------------------------------------------
# FARM SUMMARY
# ------------------------------------------------

st.header("📊 Farm Overview")

total = len(df)
low = len(df[df["Risk_Label"] == "Low"])
medium = len(df[df["Risk_Label"] == "Medium"])
high = len(df[df["Risk_Label"] == "High"])

a, b, c, d = st.columns(4)

a.metric("🐄 Total Cows", total)
b.metric("🟢 Low Risk", low)
c.metric("🟠 Medium Risk", medium)
d.metric("🔴 High Risk", high)

st.divider()

# ------------------------------------------------
# COW INFORMATION
# ------------------------------------------------

st.header("🔍 Selected Cow Information")

a, b, c, d = st.columns(4)

a.metric("Cow ID", cow["Cow_ID"])
b.metric("Breed", cow["Breed_Type"])
c.metric("Milk EC", f'{cow["Milk_EC_mS_cm"]} mS/cm')
d.metric("Milk Yield", f'{cow["Milk_Yield_L_day"]}')

# ------------------------------------------------
# SENSOR INPUT
# ------------------------------------------------

st.header("📡 Milk EC Sensor")

st.write(
    "For this prototype, the slider simulates the reading "
    "that would come from the EC sensor connected to ESP32."
)

sensor_ec = st.slider(
    "Simulated Milk EC (mS/cm)",
    min_value=3.0,
    max_value=10.0,
    value=float(cow["Milk_EC_mS_cm"]),
    step=0.1
)

st.write(f"### Current Sensor Reading: **{sensor_ec:.1f} mS/cm**")

# ------------------------------------------------
# AI PREDICTION
# ------------------------------------------------

input_data = pd.DataFrame({
    "Milk_EC_mS_cm": [sensor_ec]
})

prediction = model.predict(input_data)[0]

probabilities = model.predict_proba(input_data)[0]
classes = model.classes_

score = probabilities[
    list(classes).index(prediction)
] * 100

st.header("🤖 AI Early-Risk Assessment")

if prediction == "Low":

    st.success(
        "🟢 LOW RISK\n\n"
        "Continue routine monitoring."
    )

elif prediction == "Medium":

    st.warning(
        "🟠 MEDIUM RISK\n\n"
        "Monitor the cow closely and consider veterinary examination."
    )

else:

    st.error(
        "🔴 HIGH RISK\n\n"
        "Veterinary examination recommended."
    )

a, b = st.columns(2)

a.metric(
    "Predicted Risk",
    prediction
)

b.metric(
    "Demo Model Score",
    f"{score:.1f}%"
)

st.caption(
    "The model score is a prototype output and is NOT a "
    "clinically validated probability."
)

# ------------------------------------------------
# GRAPH
# ------------------------------------------------

st.divider()

st.header("📈 Milk EC Across Cows")

fig = px.bar(
    df,
    x="Cow_ID",
    y="Milk_EC_mS_cm",
    color="Risk_Label",
    title="Electrical Conductivity Monitoring"
)

fig.add_hline(
    y=sensor_ec,
    line_dash="dash",
    annotation_text="Current Sensor Reading"
)

st.plotly_chart(
    fig,
    use_container_width=True
)

# ------------------------------------------------
# ALERTS
# ------------------------------------------------

st.header("🚨 High-Risk Cows")

high_cows = df[df["Risk_Label"] == "High"]

st.dataframe(
    high_cows[
        [
            "Cow_ID",
            "Breed_Type",
            "Milk_EC_mS_cm",
            "SCC_thousand_cells_mL",
            "Risk_Label"
        ]
    ],
    use_container_width=True,
    hide_index=True
)

# ------------------------------------------------
# COMPLETE DATA
# ------------------------------------------------

st.header("📋 All Cow Records")

st.dataframe(
    df,
    use_container_width=True,
    hide_index=True
)

# ------------------------------------------------
# DOWNLOAD
# ------------------------------------------------

st.download_button(
    "📥 Download Farm Data",
    df.to_csv(index=False),
    "mastitis_farm_records.csv",
    "text/csv"
)

st.divider()

st.caption(
    "SIH 2026 Prototype | Synthetic demonstration dataset | "
    "EC-based early-risk indication | Not a clinical diagnosis"
)

Overwriting app.py


In [20]:
!pkill -9 -f "streamlit run" || true

^C


In [21]:
!streamlit run app.py --server.port 8503 --server.address 0.0.0.0 --server.enableCORS false --server.enableXsrfProtection false > /content/sih_dashboard.log 2>&1 &

In [22]:
from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(8503)")
print(url)

https://8503-m-s-kkb-ase1a0-1k6abnsaeb4u8-a.asia-east1-0.prod.colab.dev


In [23]:
import pandas as pd

data = {
    "Cow_ID": [
        "COW001","COW002","COW003","COW004","COW005",
        "COW006","COW007","COW008","COW009","COW010",
        "COW011","COW012","COW013","COW014","COW015",
        "COW016","COW017","COW018","COW019","COW020"
    ],

    "Breed_Type": [
        "Holstein Cross","Jersey Cross","HF Cross","Jersey Cross","HF Cross",
        "Gir Cross","HF Cross","Jersey Cross","HF Cross","Gir Cross",
        "HF Cross","Jersey Cross","HF Cross","Jersey Cross","HF Cross",
        "Gir Cross","HF Cross","Jersey Cross","HF Cross","Gir Cross"
    ],

    "Milk_EC_mS_cm": [
        4.2,4.5,4.7,4.9,5.1,
        5.0,5.6,5.8,6.0,6.1,
        6.3,6.4,6.7,6.9,7.1,
        7.3,7.5,7.7,8.0,8.2
    ],

    "Milk_Yield_L_day": [
        22.0,21.0,23.5,20.5,22.5,
        21.5,19.0,18.5,17.5,18.0,
        16.5,17.0,15.5,14.5,13.5,
        14.0,12.5,11.5,10.5,9.5
    ],

    "SCC_thousand_cells_mL": [
        145,180,165,210,195,
        225,310,360,420,390,
        480,510,620,710,820,
        760,910,980,1120,1250
    ],

    "Risk_Label": [
        "Low","Low","Low","Low","Low","Low",
        "Medium","Medium","Medium","Medium","Medium","Medium",
        "High","High","High","High","High","High","High"
    ]
}

df = pd.DataFrame(data)

df.to_csv("mastitis_20_cows.csv", index=False)

print("✅ Improved 20-cow dataset created!")
df

ValueError: All arrays must be of the same length

In [24]:
import pandas as pd

df = pd.DataFrame({
    "Cow_ID": ["COW001", "COW002", "COW003"],
    "Milk_EC_mS_cm": [4.2, 5.8, 7.2],
    "Milk_Yield_L_day": [22.0, 18.0, 12.0],
    "SCC_thousand_cells_mL": [145, 360, 910],
    "Risk_Label": ["Low", "Medium", "High"]
})

print("Dataset created successfully!")
print(df)

Dataset created successfully!
   Cow_ID  Milk_EC_mS_cm  Milk_Yield_L_day  SCC_thousand_cells_mL Risk_Label
0  COW001            4.2              22.0                    145        Low
1  COW002            5.8              18.0                    360     Medium
2  COW003            7.2              12.0                    910       High


In [25]:
import pandas as pd

data = {
    "Cow_ID": [
        "COW001","COW002","COW003","COW004","COW005",
        "COW006","COW007","COW008","COW009","COW010",
        "COW011","COW012","COW013","COW014","COW015",
        "COW016","COW017","COW018","COW019","COW020"
    ],

    "Breed_Type": [
        "Holstein Cross","Jersey Cross","HF Cross","Jersey Cross","HF Cross",
        "Gir Cross","HF Cross","Jersey Cross","HF Cross","Gir Cross",
        "HF Cross","Jersey Cross","HF Cross","Jersey Cross","HF Cross",
        "Gir Cross","HF Cross","Jersey Cross","HF Cross","Gir Cross"
    ],

    "Milk_EC_mS_cm": [
        4.2,4.5,4.7,4.9,5.1,
        5.0,5.6,5.8,6.0,6.1,
        6.3,6.4,6.7,6.9,7.1,
        7.3,7.5,7.7,8.0,8.2
    ],

    "Milk_Yield_L_day": [
        22.0,21.0,23.5,20.5,22.5,
        21.5,19.0,18.5,17.5,18.0,
        16.5,17.0,15.5,14.5,13.5,
        14.0,12.5,11.5,10.5,9.5
    ],

    "SCC_thousand_cells_mL": [
        145,180,165,210,195,
        225,310,360,420,390,
        480,510,620,710,820,
        760,910,980,1120,1250
    ],

    "Risk_Label": [
        "Low","Low","Low","Low","Low","Low",
        "Medium","Medium","Medium","Medium","Medium","Medium",
        "High","High","High","High","High","High","High","High"
    ]
}

df = pd.DataFrame(data)

df.to_csv("mastitis_20_cows.csv", index=False)

print("✅ Phase 2 dataset created successfully!")
print("Number of cows:", len(df))

df

✅ Phase 2 dataset created successfully!
Number of cows: 20


,Cow_ID,Breed_Type,Milk_EC_mS_cm,Milk_Yield_L_day,SCC_thousand_cells_mL,Risk_Label
0,COW001,Holstein Cross,4.2,22.0,145,Low
1,COW002,Jersey Cross,4.5,21.0,180,Low
2,COW003,HF Cross,4.7,23.5,165,Low
3,COW004,Jersey Cross,4.9,20.5,210,Low
4,COW005,HF Cross,5.1,22.5,195,Low
5,COW006,Gir Cross,5.0,21.5,225,Low
6,COW007,HF Cross,5.6,19.0,310,Medium
7,COW008,Jersey Cross,5.8,18.5,360,Medium
8,COW009,HF Cross,6.0,17.5,420,Medium
9,COW010,Gir Cross,6.1,18.0,390,Medium


In [26]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

# Features
X = df[
    [
        "Milk_EC_mS_cm",
        "Milk_Yield_L_day",
        "SCC_thousand_cells_mL"
    ]
]

# Target
y = df["Risk_Label"]

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Create model
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Train
model.fit(X_train, y_train)

# Test
predictions = model.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, predictions)

print("✅ Improved AI model trained successfully!")
print("Demo accuracy:", round(accuracy * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(y_test, predictions))

✅ Improved AI model trained successfully!
Demo accuracy: 100.0 %

Classification Report:
              precision    recall  f1-score   support

        High       1.00      1.00      1.00         2
         Low       1.00      1.00      1.00         2
      Medium       1.00      1.00      1.00         1

    accuracy                           1.00         5
   macro avg       1.00      1.00      1.00         5
weighted avg       1.00      1.00      1.00         5



In [27]:
import joblib

joblib.dump(model, "mastitis_model_phase2.pkl")

print("✅ Phase 2 AI model saved successfully!")
print("File: mastitis_model_phase2.pkl")

✅ Phase 2 AI model saved successfully!
File: mastitis_model_phase2.pkl


In [29]:
import pandas as pd

new_cow = pd.DataFrame({
    "Milk_EC_mS_cm": [6.8],
    "Milk_Yield_L_day": [14.0],
    "SCC_thousand_cells_mL": [750]
})

prediction = model.predict(new_cow)[0]

probabilities = model.predict_proba(new_cow)[0]
classes = model.classes_

score = probabilities[list(classes).index(prediction)] * 100

print("🐄 New Cow Prediction")
print("----------------------")
print("Milk EC:", new_cow["Milk_EC_mS_cm"].iloc[0], "mS/cm")
print("Milk Yield:", new_cow["Milk_Yield_L_day"].iloc[0], "L/day")
print("SCC:", new_cow["SCC_thousand_cells_mL"].iloc[0], "thousand cells/mL")
print("Predicted Risk:", prediction)
print("Demo Model Score:", round(score, 1), "%")

🐄 New Cow Prediction
----------------------
Milk EC: 6.8 mS/cm
Milk Yield: 14.0 L/day
SCC: 750 thousand cells/mL
Predicted Risk: High
Demo Model Score: 97.0 %


In [30]:
%%writefile app.py

import streamlit as st
import pandas as pd
import joblib
import plotly.express as px

# -----------------------------
# PAGE CONFIGURATION
# -----------------------------
st.set_page_config(
    page_title="Bovine Mastitis AI",
    page_icon="🐄",
    layout="wide"
)

# -----------------------------
# LOAD DATA AND MODEL
# -----------------------------
df = pd.read_csv("mastitis_20_cows.csv")
model = joblib.load("mastitis_model_phase2.pkl")

# -----------------------------
# HEADER
# -----------------------------
st.title("🐄 Bovine Mastitis AI")
st.subheader("AI-Based Early-Risk Monitoring System for Dairy Cows")

st.info(
    "🧪 DEMO MODE: The displayed cow measurements are "
    "synthetic demonstration values. In the final system, "
    "real sensor and farm data will be used."
)

# -----------------------------
# COW SELECTION
# -----------------------------
st.header("🐄 Select Cow")

selected_cow = st.selectbox(
    "Choose Cow ID",
    df["Cow_ID"].tolist()
)

cow = df[df["Cow_ID"] == selected_cow].iloc[0]

st.success(f"Currently monitoring: **{selected_cow}**")

# -----------------------------
# FARM OVERVIEW
# -----------------------------
st.header("📊 Farm Overview")

total = len(df)
low = len(df[df["Risk_Label"] == "Low"])
medium = len(df[df["Risk_Label"] == "Medium"])
high = len(df[df["Risk_Label"] == "High"])

a, b, c, d = st.columns(4)

a.metric("🐄 Total Cows", total)
b.metric("🟢 Low Risk", low)
c.metric("🟠 Medium Risk", medium)
d.metric("🔴 High Risk", high)

st.divider()

# -----------------------------
# SELECTED COW INFORMATION
# -----------------------------
st.header("🔍 Cow Information")

a, b, c, d = st.columns(4)

a.metric("Cow ID", cow["Cow_ID"])
b.metric("Breed", cow["Breed_Type"])
c.metric(
    "Milk EC",
    f'{cow["Milk_EC_mS_cm"]:.1f} mS/cm'
)
d.metric(
    "Milk Yield",
    f'{cow["Milk_Yield_L_day"]:.1f} L/day'
)

st.metric(
    "SCC",
    f'{cow["SCC_thousand_cells_mL"]:.0f} thousand cells/mL'
)

# -----------------------------
# SIMULATED SENSOR INPUTS
# -----------------------------
st.header("📡 Simulated Sensor / Farm Inputs")

st.write(
    "These controls simulate measurements that will "
    "eventually come from sensors and farm records."
)

sensor_ec = st.slider(
    "Milk Electrical Conductivity (mS/cm)",
    min_value=3.0,
    max_value=10.0,
    value=float(cow["Milk_EC_mS_cm"]),
    step=0.1
)

sensor_yield = st.slider(
    "Milk Yield (L/day)",
    min_value=5.0,
    max_value=30.0,
    value=float(cow["Milk_Yield_L_day"]),
    step=0.5
)

sensor_scc = st.slider(
    "SCC (thousand cells/mL)",
    min_value=100,
    max_value=1500,
    value=int(cow["SCC_thousand_cells_mL"]),
    step=10
)

# -----------------------------
# AI PREDICTION
# -----------------------------
input_data = pd.DataFrame({
    "Milk_EC_mS_cm": [sensor_ec],
    "Milk_Yield_L_day": [sensor_yield],
    "SCC_thousand_cells_mL": [sensor_scc]
})

prediction = model.predict(input_data)[0]

probabilities = model.predict_proba(input_data)[0]
classes = model.classes_

score = probabilities[
    list(classes).index(prediction)
] * 100

# -----------------------------
# AI RESULT
# -----------------------------
st.divider()

st.header("🤖 AI Early-Risk Assessment")

if prediction == "Low":

    st.success(
        "🟢 LOW RISK\n\n"
        "Continue routine monitoring."
    )

elif prediction == "Medium":

    st.warning(
        "🟠 MEDIUM RISK\n\n"
        "Monitor the cow closely and consider "
        "veterinary examination."
    )

else:

    st.error(
        "🔴 HIGH RISK\n\n"
        "Veterinary examination recommended."
    )

a, b = st.columns(2)

a.metric(
    "Predicted Risk",
    prediction
)

b.metric(
    "Demo Model Score",
    f"{score:.1f}%"
)

st.caption(
    "⚠️ The model score is a prototype output and "
    "is NOT a clinically validated probability."
)

# -----------------------------
# EC GRAPH
# -----------------------------
st.divider()

st.header("📈 Milk EC Across Cows")

fig1 = px.bar(
    df,
    x="Cow_ID",
    y="Milk_EC_mS_cm",
    color="Risk_Label",
    title="Milk Electrical Conductivity"
)

fig1.add_hline(
    y

Overwriting app.py


In [31]:
!pkill -9 -f "streamlit run" || true

^C


In [32]:
!streamlit run app.py --server.port 8503 --server.address 0.0.0.0 --server.enableCORS false --server.enableXsrfProtection false > /content/sih_dashboard.log 2>&1 &

In [33]:
from google.colab.output import eval_js

url = eval_js("google.colab.kernel.proxyPort(8503)")
print(url)

https://8503-m-s-kkb-ase1a0-1k6abnsaeb4u8-a.asia-east1-0.prod.colab.dev


In [34]:
!sed -n '195,215p' app.py

fig1 = px.bar(
    df,
    x="Cow_ID",
    y="Milk_EC_mS_cm",
    color="Risk_Label",
    title="Milk Electrical Conductivity"
)

fig1.add_hline(
    y


In [35]:
%%writefile app.py

import streamlit as st
import pandas as pd
import joblib
import plotly.express as px

st.set_page_config(page_title="Bovine Mastitis AI", page_icon="🐄", layout="wide")

df = pd.read_csv("mastitis_20_cows.csv")
model = joblib.load("mastitis_model_phase2.pkl")

st.title("🐄 Bovine Mastitis AI")
st.subheader("AI-Based Early-Risk Monitoring System")

st.info(
    "Demo prototype using synthetic cow data. "
    "The system provides an early-risk indication, not a clinical diagnosis."
)

# Select cow
st.header("🐄 Select Cow")

selected_cow = st.selectbox(
    "Choose Cow ID",
    df["Cow_ID"].tolist()
)

cow = df[df["Cow_ID"] == selected_cow].iloc[0]

st.success(f"Currently monitoring: {selected_cow}")

# Farm overview
st.header("📊 Farm Overview")

total = len(df)
low = len(df[df["Risk_Label"] == "Low"])
medium = len(df[df["Risk_Label"] == "Medium"])
high = len(df[df["Risk_Label"] == "High"])

a, b, c, d = st.columns(4)

a.metric("Total Cows", total)
b.metric("🟢 Low Risk", low)
c.metric("🟠 Medium Risk", medium)
d.metric("🔴 High Risk", high)

st.divider()

# Cow information
st.header("🔍 Selected Cow Information")

a, b, c, d = st.columns(4)

a.metric("Cow ID", cow["Cow_ID"])
b.metric("Breed", cow["Breed_Type"])
c.metric("Milk EC", f'{cow["Milk_EC_mS_cm"]} mS/cm')
d.metric("Milk Yield", f'{cow["Milk_Yield_L_day"]} L/day')

# Sensor
st.header("📡 Milk EC Sensor")

st.write(
    "This slider simulates the value that would come from "
    "an EC sensor connected to ESP32."
)

sensor_ec = st.slider(
    "Simulated Milk EC (mS/cm)",
    3.0,
    10.0,
    float(cow["Milk_EC_mS_cm"]),
    0.1
)

st.write(f"### Current Sensor Reading: {sensor_ec:.1f} mS/cm")

# AI prediction
input_data = pd.DataFrame({
    "Milk_EC_mS_cm": [sensor_ec],
    "Milk_Yield_L_day": [float(cow["Milk_Yield_L_day"])],
    "SCC_thousand_cells_mL": [float(cow["SCC_thousand_cells_mL"])]
})

prediction = model.predict(input_data)[0]

probabilities = model.predict_proba(input_data)[0]
classes = model.classes_

score = probabilities[list(classes).index(prediction)] * 100

# Result
st.header("🤖 AI Early-Risk Assessment")

if prediction == "Low":
    st.success("🟢 LOW RISK — Continue routine monitoring.")

elif prediction == "Medium":
    st.warning(
        "🟠 MEDIUM RISK — Monitor the cow closely "
        "and consider veterinary examination."
    )

else:
    st.error(
        "🔴 HIGH RISK — Veterinary examination recommended."
    )

a, b = st.columns(2)

a.metric("Predicted Risk", prediction)
b.metric("Prototype Model Score", f"{score:.1f}%")

st.caption(
    "The model score is a prototype output and is NOT a clinically "
    "validated probability."
)

st.divider()

# Graph
st.header("📈 Milk EC Across Cows")

fig1 = px.bar(
    df,
    x="Cow_ID",
    y="Milk_EC_mS_cm",
    color="Risk_Label",
    title="Milk Electrical Conductivity"
)

fig1.add_hline(
    y=sensor_ec,
    line_dash="dash",
    annotation_text="Current Sensor Reading"
)

st.plotly_chart(fig1, use_container_width=True)

# High risk cows
st.header("🚨 High-Risk Cows")

high_cows = df[df["Risk_Label"] == "High"]

st.dataframe(
    high_cows[
        [
            "Cow_ID",
            "Breed_Type",
            "Milk_EC_mS_cm",
            "Milk_Yield_L_day",
            "SCC_thousand_cells_mL",
            "Risk_Label"
        ]
    ],
    use_container_width=True,
    hide_index=True
)

# All records
st.header("📋 All Cow Records")

st.dataframe(
    df,
    use_container_width=True,
    hide_index=True
)

# Download
st.download_button(
    "📥 Download Farm Data",
    df.to_csv(index=False),
    "mastitis_farm_records.csv",
    "text/csv"
)

st.divider()

st.caption(
    "SIH 2026 Prototype | Synthetic demonstration dataset | "
    "Early-risk indication | Not a clinical diagnosis"
)

Overwriting app.py


In [36]:
import py_compile
py_compile.compile("app.py", doraise=True)
print("✅ app.py is correct!")

✅ app.py is correct!


In [37]:
!pkill -9 -f "streamlit run" || true

^C


In [38]:
!streamlit run app.py --server.port 8503 --server.address 0.0.0.0 --server.enableCORS false --server.enableXsrfProtection false > /content/sih_dashboard.log 2>&1 &

In [39]:
from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(8503)")
print(url)

https://8503-m-s-kkb-ase1a0-1k6abnsaeb4u8-a.asia-east1-0.prod.colab.dev


In [45]:
%%writefile app.py

import streamlit as st
import pandas as pd
import joblib
import plotly.express as px

st.set_page_config(
    page_title="Bovine Mastitis AI",
    page_icon="🐄",
    layout="wide"
)

# Load data and EC-only AI model
df = pd.read_csv("mastitis_20_cows.csv")
model = joblib.load("mastitis_model.pkl")

st.title("🐄 Bovine Mastitis AI")
st.subheader("AI-Based Early-Risk Monitoring System")

st.info(
    "This prototype uses a simulated EC sensor reading and synthetic "
    "demonstration data. It provides an early-risk indication, not a diagnosis."
)

# -----------------------------
# COW SELECTION
# -----------------------------

st.header("🐄 Select Cow")

selected_cow = st.selectbox(
    "Choose Cow ID",
    df["Cow_ID"].tolist()
)

cow = df[df["Cow_ID"] == selected_cow].iloc[0]

st.success(f"Currently monitoring: {selected_cow}")

# -----------------------------
# FARM OVERVIEW
# -----------------------------

st.header("📊 Farm Overview")

total = len(df)
low = len(df[df["Risk_Label"] == "Low"])
medium = len(df[df["Risk_Label"] == "Medium"])
high = len(df[df["Risk_Label"] == "High"])

a, b, c, d = st.columns(4)

a.metric("🐄 Total Cows", total)
b.metric("🟢 Low Risk", low)
c.metric("🟠 Medium Risk", medium)
d.metric("🔴 High Risk", high)

st.divider()

# -----------------------------
# SELECTED COW INFORMATION
# -----------------------------

st.header("🔍 Selected Cow Information")

a, b, c = st.columns(3)

a.metric("Cow ID", cow["Cow_ID"])
b.metric("Breed", cow["Breed_Type"])
c.metric("Recorded EC", f'{cow["Milk_EC_mS_cm"]} mS/cm')

# -----------------------------
# EC SENSOR
# -----------------------------

st.header("📡 EC Sensor Monitoring")

st.success("🟢 EC Sensor: ACTIVE (DEMO)")

sensor_ec = st.slider(
    "EC Sensor Reading (mS/cm)",
    min_value=3.0,
    max_value=10.0,
    value=float(cow["Milk_EC_mS_cm"]),
    step=0.1
)

st.metric(
    "Current EC Sensor Reading",
    f"{sensor_ec:.1f} mS/cm"
)

st.caption(
    "The slider simulates the EC value that will later come "
    "from the EC sensor connected to ESP32."
)

# -----------------------------
# AI PREDICTION
# -----------------------------

input_data = pd.DataFrame({
    "Milk_EC_mS_cm": [sensor_ec]
})

prediction = model.predict(input_data)[0]

probabilities = model.predict_proba(input_data)[0]
classes = model.classes_

score = probabilities[
    list(classes).index(prediction)
] * 100

# -----------------------------
# AI RESULT
# -----------------------------

st.header("🤖 AI Early-Risk Assessment")

if prediction == "Low":

    st.success(
        "🟢 LOW RISK\n\n"
        "Continue routine monitoring."
    )

elif prediction == "Medium":

    st.warning(
        "🟠 MEDIUM RISK\n\n"
        "Monitor the cow closely and consider veterinary examination."
    )

else:

    st.error(
        "🔴 HIGH RISK\n\n"
        "Veterinary examination recommended."
    )

a, b = st.columns(2)

a.metric("Predicted Risk", prediction)

b.metric(
    "Prototype Model Score",
    f"{score:.1f}%"
)

st.caption(
    "The model score is a prototype output and is NOT a clinically "
    "validated probability."
)

# -----------------------------
# EC GRAPH
# -----------------------------

st.divider()

st.header("📈 Milk EC Across Cows")

fig1 = px.bar(
    df,
    x="Cow_ID",
    y="Milk_EC_mS_cm",
    color="Risk_Label",
    title="Milk Electrical Conductivity"
)

fig1.add_hline(
    y=sensor_ec,
    line_dash="dash",
    annotation_text="Current Sensor Reading"
)

st.plotly_chart(
    fig1,
    use_container_width=True
)

# -----------------------------
# HIGH-RISK COWS
# -----------------------------

st.header("🚨 High-Risk Cows")

high_cows = df[df["Risk_Label"] == "High"]

st.dataframe(
    high_cows[
        [
            "Cow_ID",
            "Breed_Type",
            "Milk_EC_mS_cm",
            "Risk_Label"
        ]
    ],
    use_container_width=True,
    hide_index=True
)

# -----------------------------
# ALL RECORDS
# -----------------------------

st.header("📋 All Cow Records")

st.dataframe(
    df,
    use_container_width=True,
    hide_index=True
)

# -----------------------------
# DOWNLOAD
# -----------------------------

st.download_button(
    "📥 Download Farm Data",
    df.to_csv(index=False),
    "mastitis_farm_records.csv",
    "text/csv"
)

st.divider()

st.caption(
    "SIH 2026 Prototype | Synthetic demonstration dataset | "
    "EC-based early-risk indication | Not a clinical diagnosis"
)

Overwriting app.py


In [46]:
import py_compile
py_compile.compile("app.py", doraise=True)
print("✅ app.py is correct!")

✅ app.py is correct!


In [47]:
!pkill -9 -f "streamlit run" || true

^C


In [48]:
!streamlit run app.py --server.port 8503 --server.address 0.0.0.0 --server.enableCORS false --server.enableXsrfProtection false > /content/sih_dashboard.log 2>&1 &

In [53]:
from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(8503)")
print(url)

https://8503-m-s-kkb-ase1a0-1k6abnsaeb4u8-a.asia-east1-0.prod.colab.dev


In [50]:
!cp app.py /content/app.py
print("✅ Dashboard saved: app.py")

cp: 'app.py' and '/content/app.py' are the same file
✅ Dashboard saved: app.py


In [51]:
import os

files = [
    "app.py",
    "mastitis_model.pkl",
    "mastitis_20_cows.csv"
]

for file in files:
    if os.path.exists(file):
        print("✅", file)
    else:
        print("❌ Missing:", file)

✅ app.py
✅ mastitis_model.pkl
✅ mastitis_20_cows.csv


In [52]:
!zip -j /content/Bovine_Mastitis_SIH_Final_80pct.zip \
    /content/app.py \
    /content/mastitis_model.pkl \
    /content/mastitis_20_cows.csv

print("✅ Final 80% prototype backup created!")

  adding: app.py (deflated 62%)
  adding: mastitis_model.pkl (deflated 94%)
  adding: mastitis_20_cows.csv (deflated 59%)
✅ Final 80% prototype backup created!


# New Section